============================================================
PYSPARK UDFs & API INTEGRATION - Interview Q&A with Code
============================================================
Swiss Re Interview Prep: Senior Application Engineer
Topic: UDFs, External API Calls, and the mapPartitions Pattern
============================================================
THIS IS THE MOST CRITICAL FILE FOR YOUR INTERVIEW!
============================================================

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import StringType
import requests
from typing import Iterator
import pandas as pd
import time



ANSWER:

UDF (User Defined Function) allows you to run custom Python code on Spark.

TYPES OF UDFs:
1. Regular UDF: Python function registered for use in DataFrames
2. Pandas UDF: Vectorized, uses Arrow for faster serialization (10-100x faster)

WHEN TO USE:
- Complex logic that can't be expressed with built-in functions
- External library calls (e.g., regex, date parsing)
- API calls (with caution!)

WHEN TO AVOID:
- Simple logic (use built-in functions instead)
- Row-by-row API calls (use mapPartitions)
- Heavy computation (consider Scala/Java UDFs)

INTERVIEW TIP: "UDFs break Catalyst optimization. I only use them when
there's no built-in function available. And I always prefer Pandas UDFs
for better performance."


In [ ]:
def basic_udf_demo():
    spark = SparkSession.builder.appName("UDFDemo").master("local[*]").getOrCreate()
    
    df = spark.createDataFrame([("CL_001",), ("RX_002",), ("OT_003",)], ["claim_id"])
    
    # REGULAR UDF (Slow - Python serialization per row)
    def extract_prefix(claim_id):
        return claim_id.split("_")[0] if claim_id else ""
    
    extract_prefix_udf = udf(extract_prefix, StringType())
    
    result = df.withColumn("prefix", extract_prefix_udf(col("claim_id")))
    result.show()
    
    spark.stop()


In [ ]:
basic_udf_demo()


ANSWER:

THE PROBLEM:
You registered get_hash_id as a UDF that calls an API for every row.

If you have 1 million rows:
- 1 million HTTP requests
- 1 million network round-trips (~100ms each = 27+ hours!)
- API rate limiting or blocking
- Executor timeouts

WHY SPARK MAKES THIS WORSE:
- Spark retries failed tasks (default 4 times)
- Each retry makes the same API calls again
- Speculative execution might duplicate calls

THE FIX:
Use `mapPartitions` or `mapInPandas` to:
- Initialize HTTP client ONCE per partition
- Batch requests if API supports it
- Implement retry logic with backoff


In [ ]:
def bad_api_udf():
    def get_hash_via_api(claim_id):
        # This runs for EVERY ROW!
        response = requests.get(f"https://api.hashify.net/hash/md4/hex?value={claim_id}")
        return response.json().get("Digest", "")
    
    return udf(get_hash_via_api, StringType())


In [ ]:
bad_api_udf()


ANSWER:

The `mapPartitions` pattern allows you to:
1. Initialize expensive resources ONCE per partition
2. Process all rows in that partition
3. Clean up resources at the end

BENEFITS:
- Connection pooling (reuse HTTP sessions)
- Batch requests (send 100 IDs at once if API supports)
- Better error handling (retry logic, dead letter queue)


In [ ]:
def correct_api_approach():
    spark = SparkSession.builder.appName("CorrectAPIDemo").master("local[*]").getOrCreate()
    
    # Sample data
    data = [(f"CL_{i:03d}",) for i in range(20)]
    df = spark.createDataFrame(data, ["claim_id"])
    
    # CORRECT APPROACH: mapInPandas (Spark 3.0+)
    def fetch_hashes_per_partition(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
        """
        This function runs ONCE per partition.
        We initialize the session once and reuse it for all rows.
        """
        # Initialize session ONCE per partition (Connection Pooling)
        session = requests.Session()
        session.headers.update({"User-Agent": "SwissRe-ETL/1.0"})
        
        for pdf in iterator:
            # pdf is a pandas DataFrame chunk
            hash_results = []
            
            for claim_id in pdf["claim_id"]:
                try:
                    # In production, you would batch these requests
                    # url = f"https://api.hashify.net/hash/md4/hex?value={claim_id}"
                    # response = session.get(url, timeout=5)
                    # hash_val = response.json().get("Digest", "")
                    
                    # For demo, simulate API response
                    time.sleep(0.01)  # Simulate 10ms latency
                    hash_val = f"hash_{claim_id}"
                    
                except Exception as e:
                    hash_val = f"ERROR: {str(e)}"
                
                hash_results.append(hash_val)
            
            # Return the original data plus the new hash column
            pdf["hash_id"] = hash_results
            yield pdf
        
        # Cleanup (if needed)
        session.close()
    
    # Apply the function
    result_schema = "claim_id string, hash_id string"
    result_df = df.mapInPandas(fetch_hashes_per_partition, schema=result_schema)
    
    print("--- Result with Hashes (Correct Approach) ---")
    result_df.show()
    
    spark.stop()


In [ ]:
correct_api_approach()


ANSWER:

If the number of unique keys is small (< 100K), you can:
1. Collect unique keys on the Driver
2. Fetch hashes in a loop on the Driver (with batching)
3. Create a DataFrame from the results
4. Broadcast join back to the main DataFrame

BENEFITS:
- No API calls from executors
- Easy to batch requests
- Easier to debug


In [ ]:
def driver_prefetch_approach():
    spark = SparkSession.builder.appName("DriverPrefetchDemo").master("local[*]").getOrCreate()
    
    # Main data
    data = [(f"CL_{i:03d}", i * 100) for i in range(10)]
    claims_df = spark.createDataFrame(data, ["claim_id", "amount"])
    
    # Step 1: Collect unique claim_ids on Driver
    unique_ids = [row.claim_id for row in claims_df.select("claim_id").distinct().collect()]
    
    # Step 2: Fetch hashes on Driver (with batching)
    def fetch_hashes_batch(ids, batch_size=5):
        """Fetch hashes in batches on the driver."""
        results = {}
        session = requests.Session()
        
        for i in range(0, len(ids), batch_size):
            batch = ids[i:i+batch_size]
            for claim_id in batch:
                # Simulate API call
                # response = session.get(f"https://api.hashify.net/hash/md4/hex?value={claim_id}")
                results[claim_id] = f"hash_{claim_id}"
        
        session.close()
        return results
    
    hash_map = fetch_hashes_batch(unique_ids)
    
    # Step 3: Create a DataFrame from the hash map
    hash_data = [(k, v) for k, v in hash_map.items()]
    hash_df = spark.createDataFrame(hash_data, ["claim_id", "hash_id"])
    
    # Step 4: Broadcast join
    from pyspark.sql.functions import broadcast
    result_df = claims_df.join(broadcast(hash_df), "claim_id", "left")
    
    print("--- Result with Broadcast Join (Driver Prefetch) ---")
    result_df.show()
    
    spark.stop()


In [ ]:
driver_prefetch_approach()


ANSWER:

Pandas UDFs (also called Vectorized UDFs) are MUCH faster than regular UDFs because:
1. They use Apache Arrow for zero-copy data transfer
2. They process data in batches (pandas Series/DataFrame)
3. They leverage NumPy/Pandas optimizations

TYPES:
- SCALAR: Input Series -> Output Series
- SCALAR_ITER: Iterator of Series -> Iterator of Series (for state)
- GROUPED_MAP: GroupBy DataFrame -> DataFrame

INTERVIEW TIP: "If I must use a UDF, I always use a Pandas UDF for 10-100x performance."


In [ ]:
def pandas_udf_demo():
    spark = SparkSession.builder.appName("PandasUDFDemo").master("local[*]").getOrCreate()
    
    df = spark.createDataFrame([(1, "hello"), (2, "world")], ["id", "text"])
    
    # Pandas UDF (Vectorized)
    @pandas_udf(StringType())
    def upper_pandas(s: pd.Series) -> pd.Series:
        return s.str.upper()
    
    result = df.withColumn("text_upper", upper_pandas(col("text")))
    result.show()
    
    spark.stop()


In [ ]:
pandas_udf_demo()

In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("Q1: Basic UDF Demo")
    print("=" * 60)
    basic_udf_demo()
    
    print("\n" + "=" * 60)
    print("Q3: Correct API Approach (mapInPandas)")
    print("=" * 60)
    correct_api_approach()
    
    print("\n" + "=" * 60)
    print("Q4: Driver Prefetch + Broadcast Join")
    print("=" * 60)
    driver_prefetch_approach()
    
    print("\n" + "=" * 60)
    print("Q5: Pandas UDF (Vectorized)")
    print("=" * 60)
    pandas_udf_demo()
